In [1]:
%%writefile split_train.py

import pandas as pd
import os

records = []
test_records = []
df = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/train.csv")
for _, row in df.iterrows():
    for i in range(1, 3):
        for col in ["positive_example_", "negative_example_"]:
            body = row[f"{col}{i}"]
            label = 1 if "positive" in col else 0
            records.append(
                {
                    "body": body,
                    "rule": row["rule"],
                    "subreddit": row["subreddit"],
                    "rule_violation": label,
                }
            )
    records.append(
        {
            "body": row["body"],
            "rule": row["rule"],
            "subreddit": row["subreddit"],
            "rule_violation": row["rule_violation"],
        }
    )
    
df = pd.DataFrame(records)
df = df.drop_duplicates(subset=["rule", "body", "rule_violation"])
df.to_csv("orig_train.csv")

records = []
df = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/test.csv")
for _, row in df.iterrows():
    for i in range(1, 3):
        for col in ["positive_example_", "negative_example_"]:
            body = row[f"{col}{i}"]
            label = 1 if "positive" in col else 0
            records.append(
                {
                    "body": body,
                    "rule": row["rule"],
                    "subreddit": row["subreddit"],
                    "rule_violation": label,
                }
            )
            
df = pd.DataFrame(records)
df = df.drop_duplicates(subset=["rule", "body", "rule_violation"])
df.to_csv("new_train.csv", index=False)

Writing split_train.py


In [2]:
%%writefile generate_paraphrase.py

#!/usr/bin/env python3
"""Generate paraphrased training data using vLLM while preserving label information."""

from __future__ import annotations

import re
import os
from typing import Dict, List, Optional

import pandas as pd
from tqdm import tqdm
from vllm import LLM, SamplingParams



# Rule-specific instructions for paraphrasing
RULE_SPECIFIC_INSTRUCTIONS = {
    "No financial advice": {
        "violation": "Rewrite this financial advice comment with different words while keeping its advisory nature about investments, taxes, or crypto.",
        "compliant": "Rewrite this comment with different words while keeping it free from specific financial advice.",
    },
    "No medical advice": {
        "violation": "Rewrite this medical advice comment with different words while keeping its diagnostic or treatment recommendation nature.",
        "compliant": "Rewrite this comment with different words while keeping it free from specific medical advice.",
    },
    "No promotion of illegal activity": {
        "violation": "Rewrite this comment with different words while keeping its promotion or encouragement of illegal activities.",
        "compliant": "Rewrite this comment with different words while keeping it legal and compliant.",
    },
    "No spoilers": {
        "violation": "Rewrite this spoiler comment with different words while keeping the reveal of important plot details.",
        "compliant": "Rewrite this comment with different words while keeping it spoiler-free.",
    },
    "No Advertising": {
        "violation": "Rewrite this promotional/advertising text with different words while keeping its spammy, promotional nature with links or product promotion.",
        "compliant": "Rewrite this comment with different words while keeping it free from advertising or promotional content.",
    },
    "No legal advice": {
        "violation": "Rewrite this legal advice comment with different words while keeping its advisory nature about legal matters.",
        "compliant": "Rewrite this comment with different words while keeping it free from specific legal advice.",
    },
    "Default": {
        "violation": "Rewrite this rule-violating comment with different words while preserving its problematic nature and rule violation.",
        "compliant": "Rewrite this compliant comment with different words while keeping it appropriate and rule-compliant.",
    },
}


def get_rule_category(rule: str) -> str:
    """Extract rule category from full rule text"""
    for rule_key in RULE_SPECIFIC_INSTRUCTIONS.keys():
        if rule_key == "Default":  # Skip the default entry in matching
            continue
        if rule_key.lower() in rule.lower():
            return rule_key
    return "Default"  # Generic fallback for unknown rules



def generate_paraphrase_prompt(body: str, rule: str, subreddit: str, rule_violation: int) -> str:
    """Generate prompt for paraphrasing with rule-specific instructions"""
    rule_category = get_rule_category(rule)
    if rule_violation == 1:
        instruction = RULE_SPECIFIC_INSTRUCTIONS[rule_category]["violation"]
    else:
        instruction = RULE_SPECIFIC_INSTRUCTIONS[rule_category]["compliant"]

    prompt = f"""{instruction}

**Context:**
- Rule: {rule}
- Original comment: {body}

**Requirements:**
- Write ONLY the rewritten version (no explanations, quotes, or metadata)
- Use different wording and sentence structure
- Keep similar length to the original
- Maintain the same relationship to the rule (violation or compliance)

Your rewritten version:"""
    return prompt


def clean_paraphrased_text(text: str, original_text: str) -> Optional[str]:
    """Clean and validate paraphrased text"""
    text = text.strip()
    unwanted_prefixes = ["Here is", "Here's", "Rewritten:", "Paraphrased:", "Output:", "Response:", "Version:", "Sure,", "Certainly,"]
    for prefix in unwanted_prefixes:
        if text.lower().startswith(prefix.lower()):
            parts = text.split(":", 1)
            if len(parts) > 1:
                text = parts[1].strip()
            else:
                words = text.split(maxsplit=1)
                if len(words) > 1:
                    text = words[1].strip()
            break

    if (text.startswith('"') and text.endswith('"')) or (text.startswith("'") and text.endswith("'")):
        text = text[1:-1].strip()

    meta_indicators = ["this is a paraphrase", "i rewrote", "as requested", "here is the rewritten", "this version", "the rewritten"]
    if any(indicator in text.lower() for indicator in meta_indicators):
        return None

    if len(text) < 4 or len(text) > 1000 or text.lower() == original_text.lower():
        return None

    return text


def build_chat_messages(user_prompt: str) -> List[Dict[str, str]]:
    """Construct chat messages for tokenizer's chat template"""
    system_prompt = (
        "You are an expert at paraphrasing text while preserving its meaning, tone, and intent. "
        "You understand Reddit content moderation rules and can maintain the same relationship to rules "
        "(violation or compliance) when rewriting comment."
    )
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def main():
    # Load training data
    train_df = pd.read_csv("new_train.csv")
    sample_frac = 1.0
    n_versions = 1
    
    # Sample data
    df_sample = train_df.sample(frac=sample_frac, random_state=65)
    print(f"Sampled {len(df_sample)} out of {len(train_df)} samples ({sample_frac:.1%})")
    if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        df_sample = df_sample.sample(n=16, random_state=65)

    # Prepare requests
    requests = []
    metadata_list = []
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Preparing prompts"):
        body = row["body"]
        prompt = generate_paraphrase_prompt(body, row["rule"], row["subreddit"], row["rule_violation"])
        messages = build_chat_messages(prompt)
        requests.append(messages)
        metadata_list.append({
            "rule": row["rule"],
            "subreddit": row["subreddit"],
            "rule_violation": row["rule_violation"],
            "original_body": body,
        })

    # Initialize vLLM
    print("\nInitializing vLLM...")
    llm = LLM(
        model="/kaggle/input/qwen-3/transformers/32b-awq/1",
        max_model_len=4096,
        tensor_parallel_size=2,
        trust_remote_code=True,
        enforce_eager=True,
        gpu_memory_utilization=0.95,
        enable_prefix_caching=True,
        max_num_seqs=32,
        dtype="half"
    )

    tokenizer = llm.get_tokenizer()
    sampling_params = SamplingParams(temperature=0.8, top_p=0.9, max_tokens=250, n=n_versions)

    # Apply chat template
    print("Applying chat template...")
    prompts = []
    for messages in requests:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        prompts.append(text)

    # Generate paraphrases
    print("Generating paraphrases...")
    all_outputs = llm.generate(prompts=prompts, sampling_params=sampling_params)

    # Collect results
    print("Collecting results...")
    augmented_samples = []
    for output, meta in zip(all_outputs, metadata_list):
        for generated_output in output.outputs:
            paraphrased = generated_output.text.strip()
            cleaned = clean_paraphrased_text(paraphrased, meta["original_body"])
            if cleaned:
                augmented_samples.append({
                    "body": cleaned,
                    "rule": meta["rule"],
                    "subreddit": meta["subreddit"],
                    "rule_violation": meta["rule_violation"],
                })

    # Save results
    output_df = pd.DataFrame(augmented_samples)
    output_df = output_df.drop_duplicates(subset=["rule", "body"]).reset_index(drop=True)
    output_df.to_csv("paraphrased_train.csv", index=False)
    print(f"\nGenerated {len(output_df)} paraphrased samples")
    print(f"Saved to: paraphrased_train.csv")

if __name__ == "__main__":
    main()

Writing generate_paraphrase.py


In [3]:
%%writefile create_train.py

import os
import time
import argparse
import warnings
import pandas as pd

def main():
    parser = argparse.ArgumentParser(description="combine two csv files and remove duplicates")
    parser.add_argument("--target", type=str)
    args = parser.parse_args()
    df1 = pd.read_csv("new_train.csv")
    df2 = pd.read_csv("orig_train.csv")
    df3 = pd.read_csv("paraphrased_train.csv")
    # df4 = pd.read_csv(args.target)
    print(f"df1: {len(df1)}, df2: {len(df2)}, df3: {len(df3)}")
    df = pd.concat([df1, df2, df3], ignore_index=True)
    print(f"combined: {len(df)}")
    df = df.drop_duplicates(subset=["rule", "body", "rule_violation"])
    print(f"after drop duplicates: {len(df)}")
    df.to_csv("train.csv", index=False)
    
if __name__ == "__main__":
    main()

Writing create_train.py


In [4]:
%%writefile patch_unsloth.py

import re
import shutil
import sys
from datetime import datetime
from pathlib import Path


# 1) importlib.metadata で配布物のパスを取得（モジュール import は不要）
def find_unsloth_paths():
    paths = []
    try:
        try:
            from importlib.metadata import distribution  # Python 3.8+
        except Exception:
            # Python <3.8 の場合は importlib_metadata（backport）にフォールバック
            from importlib_metadata import distribution  # type: ignore
        dist = distribution("unsloth")  # ここは import じゃなくメタデータ参照
        for f in dist.files or []:
            # dist.files はパッケージ内の相対パス群
            if str(f).endswith("unsloth/models/rl.py"):
                # 実体パスへ解決
                p = Path(dist.locate_file(f)).resolve()
                paths.append(p)
        # 念のため __init__.py から辿る手も試す
        for f in dist.files or []:
            if str(f).endswith("unsloth/__init__.py"):
                root = Path(dist.locate_file(f)).resolve().parent
                candidate = root / "models" / "rl.py"
                if candidate.exists():
                    paths.append(candidate.resolve())
    except Exception:
        pass

    # 2) 予備: sys.path を総当たりで探索
    if not paths:
        for sp in map(Path, sys.path):
            p = sp / "unsloth" / "models" / "rl.py"
            if p.exists():
                paths.append(p.resolve())

    # 重複除去
    uniq = []
    seen = set()
    for p in paths:
        if p not in seen:
            uniq.append(p)
            seen.add(p)
    return uniq


def patch_file(rl_py: Path) -> bool:
    src = rl_py.read_text(encoding="utf-8")

    # すでに try/except 済みならスキップ
    if "except AttributeError:" in src:
        print(f"[INFO] 既にパッチ済み: {rl_py}")
        return False

    # 置換対象の exec(...) 行を try/except に展開（インデント保持）
    pattern = re.compile(
        r'^([ \t]*)exec\(f"trl\.\{RLTrainer_name\} = created_module\.Unsloth\{RLTrainer_name\}",\s*locals\(\),\s*globals\(\)\)\s*$',
        re.MULTILINE,
    )

    def repl(m):
        indent = m.group(1)
        line = 'exec(f"trl.{RLTrainer_name} = created_module.Unsloth{RLTrainer_name}", locals(), globals())'
        return (
            f"{indent}try:\n"
            f"{indent}    {line}\n"
            f"{indent}except AttributeError:\n"
            f"{indent}    return\n"
        )

    new_src, n = pattern.subn(repl, src)
    if n == 0:
        print(f"[ERROR] 置換対象の行が見つかりませんでした: {rl_py}", file=sys.stderr)
        return False

    # バックアップ
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    backup = rl_py.with_suffix(f".py.bak.{ts}")
    shutil.copy2(rl_py, backup)
    print(f"[INFO] バックアップ作成: {backup}")

    # 書き込み
    rl_py.write_text(new_src, encoding="utf-8")
    print(f"[SUCCESS] パッチ適用: {rl_py}")
    return True


def main():
    targets = find_unsloth_paths()
    if not targets:
        print(
            "[ERROR] unsloth のインストール先が見つかりませんでした。\n"
            " - venv/conda の有効化を確認\n"
            " - `python -c \"import sys; print('\\n'.join(sys.path))\"` で探索パスを確認\n"
            " - `pip show unsloth` で Location を確認し、その配下の unsloth/models/rl.py を指定して手動パッチ\n",
            file=sys.stderr,
        )
        sys.exit(1)

    ok_any = False
    for rl_py in targets:
        if rl_py.exists():
            res = patch_file(rl_py)
            ok_any = ok_any or res

    if not ok_any:
        # 見つかったが未パッチ/不要だったケース
        print("[INFO] 変更はありませんでした。")


if __name__ == "__main__":
    main()

Writing patch_unsloth.py


In [5]:
%%writefile train.py

import os
import time
import shutil

os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import unsloth
import argparse
import warnings

import pandas as pd
from datasets import Dataset
import torch
from transformers.data.data_collator import DataCollatorForSeq2Seq
from transformers.trainer_utils import set_seed
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
import torch
from accelerate import PartialState
from transformers.trainer_utils import has_length, set_seed
from unsloth.kernels.cross_entropy_loss import Fast_CrossEntropyLoss
from torch.utils.data import SequentialSampler
import random

warnings.filterwarnings("ignore")
LABEL_MAPPING = {0: "No", 1: "Yes"}
REVERSE_MAPPING = {v: k for k, v in LABEL_MAPPING.items()}

def print_memory_usage():
    if not torch.cuda.is_available():
        print("CUDA is not available.")
        return

    for i in range(torch.cuda.device_count()):
        device = torch.device(f"cuda:{i}")
        allocated = torch.cuda.memory_allocated(i) / (1024**3)
        reserved = torch.cuda.memory_reserved(i) / (1024**3)
        print(f"GPU {i} ({torch.cuda.get_device_name(i)}):")
        print(f"  Allocated: {allocated:.2f} GB")
        print(f"  Reserved:  {reserved:.2f} GB")


def create_prompt(row: pd.Series, tokenizer, max_seq_length: int) -> str:
    """Create zero-shot prompt for binary rule violation classification"""

    def truncate_text(text, max_tokens=100):
        if pd.isna(text):
            return ""
        tokens = tokenizer.encode(str(text), add_special_tokens=False)
        if len(tokens) > max_tokens:
            tokens = tokens[:max_tokens]
            return tokenizer.decode(tokens)
        return str(text)

    prompt_template = """<|im_start|>user
Does the content violate the rule? Answer with exactly one word: Yes or No.
rule: {rule}
content: {body}<|im_end|>
<|im_start|>assistant
{label}"""
    final_prompt = prompt_template.format(
        rule=row["rule"],
        body=truncate_text(row["body"], max_seq_length - 128),
        label=LABEL_MAPPING[row["rule_violation"]],
    )
    return final_prompt


def create_datasets(train_df, max_seq_length, tokenizer):
    """Create training dataset"""
    train_examples = []
    for idx, row in train_df.iterrows():
        prompt = create_prompt(row, tokenizer, max_seq_length)
        if "\nYes" in prompt or "\nNo" in prompt:
            train_examples.append({"text": prompt, "label": row["rule_violation"]})
    train_dataset = Dataset.from_list(train_examples)
    return train_dataset


def main():
    parser = argparse.ArgumentParser(description="Train unsloth model")
    parser.add_argument("--epochs", type=float, default=1.0, help="Number of training epochs")
    parser.add_argument("--lr", type=float, default=1e-4, help="Learning rate for training")
    parser.add_argument("--max-seq-length", type=int, default=288, help="Maximum sequence length for the model")
    parser.add_argument("--model-name", type=str, default="./lora_weight", help="Model checkpoint to load for training")
    parser.add_argument("--train-csv", type=str, default="train.csv", help="CSV file containing the training data")
    parser.add_argument("--pseudo-csv", type=str, default="pseudo.csv", help="CSV file containing pseudo labels")
    parser.add_argument("--lora-r", type=int, default=16, help="LoRA rank")
    parser.add_argument("--bz", type=int, default=16, help="batch size")
    parser.add_argument("--grad-ac", type=int, default=1, help="gradient accumulation")
    parser.add_argument("--save-path", type=str)
    parser.add_argument("--seed", type=int)
    args = parser.parse_args()

    seed = args.seed
    set_seed(seed)

    model_name = args.model_name
    max_seq_length = args.max_seq_length
    train_df = pd.read_csv(args.train_csv)

    if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        train_df = train_df.sample(n=32, replace=True, random_state=42)

    # Load model and tokenizer
    print(f"Loading model: {model_name}")
    device_string = PartialState().process_index
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        device_map={"": device_string},
        load_in_4bit=True,
        load_in_8bit=False,
        dtype=torch.bfloat16 if is_bfloat16_supported() else torch.float16,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=args.lora_r,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=args.lora_r,
        lora_dropout=0.0,
        use_gradient_checkpointing="unsloth",
        random_state=seed,
    )
    tokenizer.padding_side = "right"
    
    choice_tokens = [
        tokenizer.encode("No", add_special_tokens=False)[0],
        tokenizer.encode("Yes", add_special_tokens=False)[0],
    ]
    token2choice = {token: choice for token, choice in zip(choice_tokens, ["No", "Yes"])}
    print(f"Choice tokens: {choice_tokens}")
    print(f"Token mapping: {token2choice}")
    
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        padding=True,
        return_tensors="pt",
        max_length=max_seq_length,
    )
    # Main training
    train_dataset = create_datasets(train_df, max_seq_length, tokenizer)
    print_memory_usage()
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
        args=SFTConfig(
            max_length=max_seq_length,
            per_device_train_batch_size=args.bz,
            ddp_find_unused_parameters=False,
            gradient_accumulation_steps=args.grad_ac,
            warmup_steps=5,
            num_train_epochs=args.epochs,
            fp16=True,
            logging_steps=10,
            learning_rate=args.lr,
            output_dir="outputs",
            optim="adamw_8bit",
            seed=seed,
            data_seed=seed,
            report_to=["none"],
            save_strategy="epoch",
            save_total_limit=1,
            lr_scheduler_type="cosine",
            gradient_checkpointing=True,
        ),
    )

    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
        tokenizer=tokenizer,
    )

    print("Starting training...")
    trainer.train()

    model.save_pretrained(args.save_path)
    tokenizer.save_pretrained(args.save_path)


if __name__ == "__main__":
    main()

Writing train.py


In [6]:
%%writefile save_inference_data.py

import argparse
import os

os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import pandas as pd
from transformers import AutoTokenizer

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Configure inference parameters for train_unsloth")
    parser.add_argument("--test_path", type=str, default="./test.csv", help="Path to the test CSV file")
    parser.add_argument("--max-seq-length", type=int, help="Maximum sequence length to use for tokenization")
    parser.add_argument("--model-name", type=str, help="Path or identifier for the model to load")
    parser.add_argument("--prefix", type=str, help="Prefix for output files")
    return parser.parse_args()

LABEL_MAPPING = {0: "No", 1: "Yes"}

def create_prompt(row: pd.Series, tokenizer, max_seq_length: int) -> str:
    """Create zero-shot prompt for binary rule violation classification"""

    def truncate_text(text, max_tokens=100):
        if pd.isna(text):
            return ""
        tokens = tokenizer.encode(str(text), add_special_tokens=False)
        if len(tokens) > max_tokens:
            tokens = tokens[:max_tokens]
            return tokenizer.decode(tokens)
        return str(text)

    prompt_template = """<|im_start|>user
Does the content violate the rule? Answer with exactly one word: Yes or No.
rule: {rule}
content: {body}<|im_end|>
<|im_start|>assistant
"""
    
    final_prompt = prompt_template.format(
        rule=row["rule"],
        body=truncate_text(row["body"], max_seq_length - 128),
    )
    return final_prompt

if __name__ == "__main__":
    args = parse_args()
    max_seq_length = args.max_seq_length
    model_name = args.model_name
    
    test_df = pd.read_csv(args.test_path)
    if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        test_df = test_df.sample(n=16, replace=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "right"
    
    prompts_with_length = []
    for idx, row in test_df.iterrows():
        prompt = create_prompt(row, tokenizer, max_seq_length)
        prompt_length = len(tokenizer.encode(prompt, add_special_tokens=False))
        prompts_with_length.append((prompt, prompt_length, idx))

    # Sort by length to minimize padding in batches
    test_df["prompt"] = [item[0] for item in prompts_with_length]
    test_df["prompt_length"] = [item[1] for item in prompts_with_length]
    test_df = test_df.sort_values(by="prompt_length", ascending=False).reset_index(drop=True)
    
    test_df.iloc[[i for i in range(len(test_df)) if i % 2 == 0]].to_csv(f"{args.prefix}_1.csv", index=False)
    test_df.iloc[[i for i in range(len(test_df)) if i % 2 == 1]].to_csv(f"{args.prefix}_2.csv", index=False)

Writing save_inference_data.py


In [7]:
%%writefile inference.py

import argparse
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import unsloth
import pandas as pd

# from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import numpy as np

import gc
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import time

import torch
from unsloth import FastLanguageModel
from transformers.data.data_collator import DataCollatorForSeq2Seq
from accelerate import PartialState
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Configure inference parameters for train_unsloth"
    )
    parser.add_argument(
        "--data-path",
        type=str,
        default="test.csv",
        help="Path to the input data file",
    )
    parser.add_argument(
        "--save-path",
        type=str,
        default="pred_1.csv",
        help="Path to save the output predictions",
    )
    parser.add_argument(
        "--max-seq-length",
        type=int,
        help="Maximum sequence length to use for tokenization",
    )
    parser.add_argument(
        "--batch-size",
        type=int,
        help="Batch size to use during inference",
    )
    parser.add_argument(
        "--model-name",
        type=str,
        help="Path or identifier for the model to load",
    )
    return parser.parse_args()


def inference(data_subset, model,tokenizer, batch_size: int, max_seq_length: int):
    """Batch inference function for ThreadPoolExecutor using DataCollator"""
    predictions = []
    tokenizer.padding_side = "right"
    # Set up data collator for efficient batching
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        padding=True,
        return_tensors="pt",
        max_length=max_seq_length,
    )
    # Get token ID for 'Yes' once
    yes_token_id = tokenizer.encode("Yes", add_special_tokens=False)[0]
    no_token_id = tokenizer.encode("No", add_special_tokens=False)[0]
    # Process in batches
    for i in range(0, len(data_subset), batch_size):
        batch_prompts = data_subset[i : i + batch_size]

        # Tokenize each prompt individually for data collator
        batch_inputs = []
        for prompt in batch_prompts:
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=max_seq_length,
                add_special_tokens=False,
            )
            # Convert to dictionary format for data collator
            batch_inputs.append(
                {
                    "input_ids": inputs["input_ids"].squeeze(0),
                    "attention_mask": inputs["attention_mask"].squeeze(0),
                }
            )

        # Use data collator for efficient batching
        collated_batch = data_collator(batch_inputs)
        collated_batch = {
            k: v.to(model.device) for k, v in collated_batch.items() if k != "labels"
        }

        with torch.no_grad():
            outputs = model(**collated_batch)
            logits = outputs.logits.float()  # Shape: [batch_size, seq_len, vocab_size]

        # Get probabilities for Yes/No tokens for each sample in batch
        # print(logits.shape)
        attention_mask = collated_batch["attention_mask"]
        for j in range(len(batch_prompts)):
            # Find the last non-padding token position for this sample
            last_token_pos = attention_mask[j].sum() - 1  # Last non-padding position

            # 入力トークン列から該当位置のトークンIDを取得
            input_ids = collated_batch["input_ids"][j]
            last_token_id = input_ids[last_token_pos].item()
            last_token_text = tokenizer.decode(last_token_id)
            # Get logits from the last non-padding token position
            last_token_logits = logits[j, last_token_pos, :]  # Shape: [vocab_size]

            probs = torch.softmax(last_token_logits, dim=-1)

            # Yes/Noトークンの確率を表示
            yes_prob = probs[yes_token_id].item()
            no_prob = probs[no_token_id].item()
            yes_prob = yes_prob / (yes_prob + no_prob)
            
            # # NaNチェック
            if torch.isnan(probs).any():
                print(attention_mask)

            predictions.append(yes_prob)

    return predictions


if __name__ == "__main__":
    args = parse_args()
    max_seq_length = args.max_seq_length
    batch_size = args.batch_size
    model_name = args.model_name
    df = pd.read_csv(args.data_path)
    device_string = PartialState().process_index
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        device_map={'':device_string},
        load_in_4bit=True,
        dtype=torch.float16,
    )

    model = FastLanguageModel.for_inference(model)
    

    # Run inference with ThreadPoolExecution
    start = time.time()
    predictions_sorted = inference(
        df["prompt"].tolist(), model, tokenizer, batch_size, max_seq_length
    )
    print(
        f"inference time[s]: {(time.time() - start)}, num_predictions: {len(predictions_sorted)}"
    )
    df["rule_violation"] = predictions_sorted
    df.to_csv(args.save_path, index=False)

Writing inference.py


In [8]:
%%writefile save_sub.py
import argparse
import pandas as pd


def main(args):
    # predファイル読み込み
    test_df1 = pd.read_csv(f"{args.prefix}_1.csv")
    test_df2 = pd.read_csv(f"{args.prefix}_2.csv")

    # 結合してsubmission作成
    test_df = pd.concat([test_df1, test_df2], ignore_index=True)
    submission_df = pd.DataFrame(
        {"row_id": test_df["row_id"], "rule_violation": test_df["rule_violation"]}
    )
    submission_df.to_csv(args.submission_path, index=False)
    if args.save_pseudo:
        test_df = test_df[
            (test_df["rule_violation"] > 0.97) | (test_df["rule_violation"] < 0.03)
        ].copy()
        test_df["rule_violation"] = test_df["rule_violation"].apply(round).astype(int)
        test_df = test_df.drop_duplicates(subset=["rule", "body", "rule_violation"])
        test_df[["rule", "body", "rule_violation"]].to_csv("pseudo.csv", index=False)


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Combine predictions and create submission file."
    )
    parser.add_argument(
        "--submission-path",
        type=str,
        default="submission.csv",
        help="Path to save the submission CSV file",
    )
    parser.add_argument(
        "--prefix",
        type=str,
        help="Path to save the submission CSV file",
    )
    parser.add_argument("--save-pseudo", action="store_true", help="学習を実行する")
    args = parser.parse_args()

    main(args)

Writing save_sub.py


In [9]:
%%writefile run_inference_parallel.sh

#!/usr/bin/env bash
set -euo pipefail

if [ "$#" -lt 4 ]; then
  echo "Usage: $0 <gpu0-data-path> <gpu0-save-path> <gpu1-data-path> <gpu1-save-path> [extra-args...]" >&2
  exit 1
fi

GPU0_DATA_PATH=$1
GPU0_SAVE_PATH=$2
GPU1_DATA_PATH=$3
GPU1_SAVE_PATH=$4
shift 4

EXTRA_ARGS=("$@")

CUDA_VISIBLE_DEVICES=0 UNSLOTH_DISABLE_CACHE=1 python inference.py \
  --data-path "$GPU0_DATA_PATH" \
  --save-path "$GPU0_SAVE_PATH" \
  "${EXTRA_ARGS[@]}" &
PID0=$!

sleep 20

CUDA_VISIBLE_DEVICES=1 UNSLOTH_DISABLE_CACHE=1 python inference.py \
  --data-path "$GPU1_DATA_PATH" \
  --save-path "$GPU1_SAVE_PATH" \
  "${EXTRA_ARGS[@]}" &
PID1=$!

wait "$PID0"
wait "$PID1"

Writing run_inference_parallel.sh


In [10]:
!python patch_unsloth.py
!python split_train.py
!python generate_paraphrase.py
!chmod +x run_inference_parallel.sh

[INFO] バックアップ作成: /usr/local/lib/python3.11/dist-packages/unsloth/models/rl.py.bak.20251022-150643
[SUCCESS] パッチ適用: /usr/local/lib/python3.11/dist-packages/unsloth/models/rl.py
2025-10-22 15:06:51.600753: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761145611.796881      70 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761145611.853489      70 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO 10-22 15:07:05 [__init__.py:239] Automatically detected platform cuda.
Sampled 38 out of 38 samples (100.0%)
Preparing prompts: 100%|█████████████████████| 16/16 [00:00<00:00, 10908.46it/s]

Initializing vLLM...
INFO 10-22 15:07:23 [config.py:717] This mod

In [11]:
# === パラメータ定義 ===
import os
num = "one"
model = "/kaggle/input/base/transformers/phi-4-unsloth-bnb-4bit/1"
batch_size = 4
max_seq_length = 512

# if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
!python create_train.py --target /kaggle/input/jigsaw/paraphrased_train_litellm_1.csv

# === 実行 ===
!accelerate launch train.py \
    --train-csv train.csv \
    --save-path ./lora_{num} \
    --model-name {model} \
    --lora-r 24 \
    --epochs 1 \
    --lr 2e-4 \
    --max-seq-length {max_seq_length} \
    --bz {batch_size} \
    --seed 1 \
    --grad-ac 2

!python save_inference_data.py \
    --test_path /kaggle/input/jigsaw-agile-community-rules/test.csv \
    --max-seq-length {max_seq_length} \
    --prefix {num} \
    --model-name ./lora_{num}

!./run_inference_parallel.sh \
    {num}_1.csv {num}_pred_1.csv \
    {num}_2.csv {num}_pred_2.csv \
    --batch-size {batch_size} \
    --max-seq-length {max_seq_length} \
    --model-name ./lora_{num}

!python save_sub.py --submission-path sub_{num}.csv --prefix {num}_pred
!rm -rf ./lora_{num} ./outputs

df1: 38, df2: 1886, df3: 16
combined: 1940
after drop duplicates: 1902
[W1022 15:14:13.889046414 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1022 15:14:21.892250943 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2025-10-22 15:14:32.791668: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-22 15:14:32.796130: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761146072.833156     338 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN whe

In [12]:
# === パラメータ定義 ===
import os
num = "two"
model = "/kaggle/input/base/transformers/qwen2.5-14b-unsloth-bnb-4bit/1"
batch_size = 4
max_seq_length = 512

# if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
!python create_train.py --target /kaggle/input/jigsaw/paraphrased_train_litellm_2.csv

# === 実行 ===
!accelerate launch train.py \
    --train-csv train.csv \
    --save-path ./lora_{num} \
    --model-name {model} \
    --lora-r 24 \
    --epochs 1 \
    --lr 2e-4 \
    --max-seq-length {max_seq_length} \
    --bz {batch_size} \
    --seed 2 \
    --grad-ac 2

!python save_inference_data.py \
    --test_path /kaggle/input/jigsaw-agile-community-rules/test.csv \
    --max-seq-length {max_seq_length} \
    --prefix {num} \
    --model-name ./lora_{num}

!./run_inference_parallel.sh \
    {num}_1.csv {num}_pred_1.csv \
    {num}_2.csv {num}_pred_2.csv \
    --batch-size {batch_size} \
    --max-seq-length {max_seq_length} \
    --model-name ./lora_{num}

!python save_sub.py --submission-path sub_{num}.csv --prefix {num}_pred
!rm -rf ./lora_{num} ./outputs

df1: 38, df2: 1886, df3: 16
combined: 1940
after drop duplicates: 1902
[W1022 15:20:27.455069995 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1022 15:20:35.457523632 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2025-10-22 15:20:43.574379: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761146443.608466     784 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761146443.619487     784 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registere

In [13]:
# === パラメータ定義 ===
import os
num = "three"
model = "/kaggle/input/base/transformers/qwen3-14b-unsloth-bnb-4bit/1"
batch_size = 4
max_seq_length = 512

# if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
!python create_train.py --target /kaggle/input/jigsaw/paraphrased_train_litellm_3.csv

# === 実行 ===
!accelerate launch train.py \
    --train-csv train.csv \
    --save-path ./lora_{num} \
    --model-name {model} \
    --lora-r 24 \
    --epochs 1 \
    --lr 2e-4 \
    --max-seq-length {max_seq_length} \
    --bz {batch_size} \
    --seed 3 \
    --grad-ac 2

!python save_inference_data.py \
    --test_path /kaggle/input/jigsaw-agile-community-rules/test.csv \
    --max-seq-length {max_seq_length} \
    --prefix {num} \
    --model-name ./lora_{num}

!./run_inference_parallel.sh \
    {num}_1.csv {num}_pred_1.csv \
    {num}_2.csv {num}_pred_2.csv \
    --batch-size {batch_size} \
    --max-seq-length {max_seq_length} \
    --model-name ./lora_{num}

!python save_sub.py --submission-path sub_{num}.csv --prefix {num}_pred
!rm -rf ./lora_{num} ./outputs

df1: 38, df2: 1886, df3: 16
combined: 1940
after drop duplicates: 1902
[W1022 15:26:47.822590695 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1022 15:26:55.824899253 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2025-10-22 15:27:04.137514: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761146824.179250    1166 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-22 15:27:04.180010: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT whe

In [14]:
import pandas as pd
import numpy as np
import os

# ============================================
# 各subファイルと対応するAUCスコアを定義
# ============================================
auc_scores = {
    "sub_one.csv": 1.0,
    "sub_two.csv": 1.0,
    "sub_three.csv": 1.0
}

# 読み込むファイルを抽出（存在チェック付き）
files = [f for f in auc_scores.keys() if os.path.exists(f)]
if not files and len(files) != 3:
    raise FileNotFoundError("No sub_x.csv files found!")

weights = np.array([auc_scores[f] for f in files])
weights = weights / weights.sum()

print("🔹 Weights:")
for f, w in zip(files, weights):
    print(f"  {f}: {w:.4f}")

# ============================================
# CSVを読み込み、inner joinで結合
# ============================================
dfs = []
for i, f in enumerate(files, start=1):
    df = pd.read_csv(f)
    df["row_id"] = df["row_id"].astype(int)
    df["rule_violation"] = df["rule_violation"].astype(float)
    df = df.rename(columns={"rule_violation": f"rule_violation_{i}"})
    dfs.append(df)

df = dfs[0]
for d in dfs[1:]:
    df = df.merge(d, on="row_id", how="inner")

# ============================================
# Softmax重みを適用して加重平均
# ============================================
rule_cols = [c for c in df.columns if c.startswith("rule_violation_")]
rule_matrix = df[rule_cols].values
df["rule_violation"] = np.average(rule_matrix, axis=1, weights=weights)

# 提出用ファイル出力
submission = df[["row_id", "rule_violation"]].sort_values("row_id")
submission.to_csv("submission.csv", index=False)

print(f"✅ {len(files)} 個のファイルで Softmax-weighted ensemble 完了")

🔹 Weights:
  sub_one.csv: 0.3333
  sub_two.csv: 0.3333
  sub_three.csv: 0.3333
✅ 3 個のファイルで Softmax-weighted ensemble 完了


In [15]:
import os
import shutil

def clean_directory():
    current_dir = os.getcwd()
    keep_file = "submission.csv"

    for item in os.listdir(current_dir):
        path = os.path.join(current_dir, item)
        if item == keep_file:
            continue
        if os.path.isfile(path) or os.path.islink(path):
            os.remove(path)       # ファイル or シンボリックリンクを削除
        elif os.path.isdir(path):
            shutil.rmtree(path)   # ディレクトリを削除

# if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
clean_directory()